# Gate Analysis — WideModelAttentionGated
Captures the learned sigmoid gate values `[N, C]` for every val cell and asks:
- Which markers does each cell type attend to? (mean gate heatmap)
- Is within-class gate variance high? If yes → gates cause fragmentation, not biology
- Do the fragmented UMAP islands correspond to distinct gate patterns?

In [ ]:
# ── Parameters — edit these ───────────────────────────────────────────────────
CHECKPOINT = '../z_RUNS/MIBI_TNBC_CIMATT_Gate_VICReg/last_checkpoint'
CONFIG     = '../z_RUNS/MIBI_TNBC_CIMATT_Gate_VICReg/CIMATT_Gate_VICReg.py'
H5         = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/MIBI_TNBC/MIBI_TNBC.h5'
MARKERS    = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/MIBI_TNBC/used_markers.txt'
VAL_IDX    = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/MIBI_TNBC/val.txt'
PATCH_SIZE = 32
BATCH_SIZE = 256
DEVICE     = 'cuda'   # or 'cpu'

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from copy import deepcopy

import src   # registers MODELS / DATASETS / HOOKS
from src.models import WideModelAttentionGated
from mmengine.config import Config
from mmengine.registry import MODELS, DATASETS
from torch.utils.data import DataLoader

## 1 — Load model & patch forward to capture gates

In [ ]:
cfg = Config.fromfile(CONFIG)
model = MODELS.build(cfg.model)

# resolve last_checkpoint text file → actual .pth path
ckpt_path = Path(CHECKPOINT)
if ckpt_path.suffix == '':
    ckpt_path = Path(ckpt_path.read_text().strip())

state = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(state.get('state_dict', state), strict=False)
model.to(DEVICE).eval()
print('Model loaded.')

In [ ]:
# Find the backbone
backbone = next(m for m in model.modules() if isinstance(m, WideModelAttentionGated))

captured_gates = []
original_forward = backbone.forward

def forward_with_gate_capture(x, *args, **kwargs):
    import torch.nn.functional as F
    if backbone.input_norm:
        x = F.normalize(x, dim=1)
    x = backbone.stem(x)
    x = backbone.layers(x)
    B, CD, H, W = x.shape
    C, D = backbone.in_channels, backbone.stem_width
    tokens = x.view(B, C, D, H, W).mean(dim=(-2, -1))
    rel    = tokens - tokens.mean(dim=1, keepdim=True)
    gates  = torch.sigmoid(
        backbone.gate_proj(rel).squeeze(-1)
        / backbone.gate_temp.abs().clamp(min=1e-4)
    )                                                   # [B, C]
    captured_gates.append(gates.detach().cpu())
    # continue normally
    tokens = tokens * gates.unsqueeze(-1)
    tokens_t = backbone.attn_norm(tokens).transpose(0, 1)
    attn_out, _ = backbone.channel_attn(tokens_t, tokens_t, tokens_t)
    tokens = tokens + attn_out.transpose(0, 1)
    tokens = tokens + backbone.ffn(backbone.ffn_norm(tokens))
    correction = tokens.view(B, CD, 1, 1)
    x = x + correction
    return (x.mean(dim=(-2, -1)).view(B, CD, 1, 1),)

backbone.forward = forward_with_gate_capture
print('Forward patched — gates will be captured during inference.')

## 2 — Build val dataloader

In [ ]:
marker_names = Path(MARKERS).read_text().strip().splitlines()
print(f'{len(marker_names)} markers:', marker_names)

val_pipeline = [
    dict(type='CenterCrop', size=PATCH_SIZE),
    dict(type='ToTensor'),
]

val_dataset = DATASETS.build(dict(
    type='MCIDataset',
    h5_filepath=H5,
    patch_size=PATCH_SIZE,
    used_markers=MARKERS,
    used_indicies=VAL_IDX,
    ignore_annotation=['Unidentified'],
    pipeline=val_pipeline,
))

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, num_workers=8,
    shuffle=False, drop_last=False,
    collate_fn=lambda b: b,
)
print(f'Val set: {len(val_dataset)} cells')

## 3 — Run inference (gates captured as side-effect)

In [ ]:
all_labels, all_sample_ids, all_feats = [], [], []
captured_gates.clear()

with torch.no_grad():
    for batch in tqdm(val_loader):
        imgs = torch.stack([b['inputs'][0] for b in batch]).float().to(DEVICE)
        feats = model([imgs], mode='tensor')[0].squeeze()
        all_feats.append(feats.cpu().numpy())
        all_labels.extend([b['data_samples'].annotation  for b in batch])
        all_sample_ids.extend([b['data_samples'].sample_id for b in batch])

gates      = torch.cat(captured_gates, dim=0).numpy()   # [N, C]
all_feats  = np.concatenate(all_feats, axis=0)           # [N, C*D]
all_labels = np.array(all_labels)
all_sids   = np.array(all_sample_ids)

print(f'Gates: {gates.shape}  |  Feats: {all_feats.shape}  |  Labels: {all_labels.shape}')

## 4 — Mean gate heatmap per cell type
**What to look for:** Do different cell types show distinct gate patterns (high for their defining markers)?
If yes → gating reflects biology.

In [ ]:
class_names = sorted(set(all_labels))
mean_gates  = np.stack([gates[all_labels == c].mean(0) for c in class_names])  # [n_classes, C]

fig, ax = plt.subplots(figsize=(max(14, len(marker_names) * 0.45), max(5, len(class_names) * 0.5)))
sns.heatmap(
    mean_gates, xticklabels=marker_names, yticklabels=class_names,
    cmap='viridis', vmin=0, vmax=1, linewidths=0.3, ax=ax
)
ax.set_title('Mean sigmoid gate per cell type per marker', fontsize=13)
ax.set_xlabel('Marker'); ax.set_ylabel('Cell type')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 5 — Within-class gate std heatmap
**What to look for:** High std within a cell type for a marker → that marker's gate is inconsistent within the class → **fragmentation driver**.
Low std everywhere → gates are stable within classes → fragmentation has another cause.

In [ ]:
std_gates = np.stack([gates[all_labels == c].std(0) for c in class_names])

fig, ax = plt.subplots(figsize=(max(14, len(marker_names) * 0.45), max(5, len(class_names) * 0.5)))
sns.heatmap(
    std_gates, xticklabels=marker_names, yticklabels=class_names,
    cmap='rocket_r', linewidths=0.3, ax=ax
)
ax.set_title('Within-class gate std per marker  (high = fragmentation driver)', fontsize=13)
ax.set_xlabel('Marker'); ax.set_ylabel('Cell type')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 6 — Gate boxplot for a specific marker
Pick any marker of interest (e.g. CD4, CD8, PanCK) and see its gate distribution per cell type.

In [ ]:
# Top-8 markers by overall gate variance (most informative to inspect)
top_markers = np.argsort(gates.var(0))[::-1][:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, mi in zip(axes.flat, top_markers):
    data = [gates[all_labels == c, mi] for c in class_names]
    ax.boxplot(data, labels=class_names, patch_artist=True,
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(marker_names[mi], fontsize=11)
    ax.set_ylabel('gate value')
    ax.set_ylim(0, 1)
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right', fontsize=7)

plt.suptitle('Gate distributions per cell type — top-8 highest-variance markers', fontsize=13)
plt.tight_layout()
plt.show()

## 7 — UMAP coloured by gate value
**What to look for:** If fragmented islands each have a distinct, uniform colour → gate values drive the fragmentation.
If colours are mixed within islands → something else is fragmenting.

In [ ]:
from umap import UMAP

print('Computing UMAP …')
umap_coords = UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42).fit_transform(all_feats)
print('Done.')

In [ ]:
# UMAP coloured by cell type (reference)
from matplotlib.colors import to_rgba
palette = plt.cm.tab20.colors
color_map = {c: palette[i % len(palette)] for i, c in enumerate(class_names)}
colors = [color_map[l] for l in all_labels]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(umap_coords[:, 0], umap_coords[:, 1], c=colors, s=1, alpha=0.4)
handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=color_map[c], markersize=7, label=c) for c in class_names]
ax.legend(handles=handles, fontsize=7, loc='best', ncol=2)
ax.set_title('UMAP — cell type'); ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
plt.tight_layout(); plt.show()

In [ ]:
# UMAP coloured by gate value for top-variance markers
n_show = min(9, len(top_markers))
ncols = 3
nrows = (n_show + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 5))
for ax, mi in zip(np.array(axes).flat, top_markers[:n_show]):
    sc = ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
                    c=gates[:, mi], cmap='plasma', s=1, alpha=0.5, vmin=0, vmax=1)
    plt.colorbar(sc, ax=ax, label='gate')
    ax.set_title(f'{marker_names[mi]}', fontsize=11)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

# hide unused axes
for ax in np.array(axes).flat[n_show:]:
    ax.set_visible(False)

plt.suptitle('UMAP coloured by gate value — top high-variance markers', fontsize=13)
plt.tight_layout(); plt.show()

## 8 — Are gate patterns sample-specific?
Check if gate variance is explained more by sample ID than by cell type. If yes → gates still have a sample-level component.

In [ ]:
sample_ids = sorted(set(all_sids))
mean_gates_sample = np.stack([gates[all_sids == s].mean(0) for s in sample_ids])

fig, ax = plt.subplots(figsize=(max(14, len(marker_names) * 0.45), max(4, len(sample_ids) * 0.4)))
sns.heatmap(
    mean_gates_sample, xticklabels=marker_names, yticklabels=sample_ids,
    cmap='viridis', vmin=0, vmax=1, linewidths=0.3, ax=ax
)
ax.set_title('Mean gate per sample per marker  (high variance → gates still sample-specific)', fontsize=13)
ax.set_xlabel('Marker'); ax.set_ylabel('Sample ID')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 9 — Variance decomposition: cell type vs sample vs residual
For each marker gate, compute how much variance is explained by cell type, sample, and unexplained (residual = potential fragmentation noise).

In [ ]:
var_celltype, var_sample, var_residual = [], [], []

for mi in range(gates.shape[1]):
    g = gates[:, mi]
    grand_mean = g.mean()

    # cell-type explained variance
    ct_means  = np.array([g[all_labels == c].mean() for c in class_names])
    ct_counts = np.array([( all_labels == c).sum()  for c in class_names])
    ss_ct = float(np.sum(ct_counts * (ct_means - grand_mean) ** 2))

    # sample explained variance
    s_means  = np.array([g[all_sids == s].mean() for s in sample_ids])
    s_counts = np.array([(all_sids == s).sum()   for s in sample_ids])
    ss_s  = float(np.sum(s_counts * (s_means - grand_mean) ** 2))

    ss_tot = float(((g - grand_mean) ** 2).sum())
    ss_res = max(ss_tot - ss_ct - ss_s, 0)

    var_celltype.append(ss_ct / ss_tot if ss_tot > 0 else 0)
    var_sample.append(ss_s  / ss_tot if ss_tot > 0 else 0)
    var_residual.append(ss_res / ss_tot if ss_tot > 0 else 0)

var_df = np.array([var_celltype, var_sample, var_residual])  # [3, C]

fig, ax = plt.subplots(figsize=(max(14, len(marker_names) * 0.45), 4))
x = np.arange(len(marker_names))
ax.bar(x, var_celltype, label='Cell type', color='steelblue')
ax.bar(x, var_sample,   label='Sample',    color='tomato',    bottom=var_celltype)
bottom2 = np.array(var_celltype) + np.array(var_sample)
ax.bar(x, var_residual, label='Residual',  color='lightgrey', bottom=bottom2)
ax.set_xticks(x); ax.set_xticklabels(marker_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Fraction of gate variance'); ax.set_ylim(0, 1)
ax.set_title('Gate variance decomposition per marker')
ax.legend()
plt.tight_layout(); plt.show()

print(f'Mean cell-type variance explained: {np.mean(var_celltype):.2%}')
print(f'Mean sample    variance explained: {np.mean(var_sample):.2%}')
print(f'Mean residual  variance:           {np.mean(var_residual):.2%}')